# Trade-Off Questions: "Why X Over Y?"

A large class of MLE interview questions is: "Why would you use X instead of Y?" or "What's the tradeoff between X and Y?" This note organizes 25 such questions into themes with structured answers you can adapt in real interviews.

## What Interviewers Test
- Depth of understanding: do you know *why*, not just *what*?
- Nuanced thinking: most tradeoffs are contextual, not absolute
- Communication: can you explain a tradeoff in 2–3 sentences?
- Production awareness: do you consider serving cost, not just model accuracy?

## Theme 1: Model Selection

**Q1: Why use gradient boosting (GBDT) over a neural network for tabular data?**
GBDT (XGBoost, LightGBM) typically outperforms neural networks on tabular data with hundreds to thousands of features. It handles mixed feature types (categorical, numerical) natively, is robust to outliers, requires no feature normalization, and trains in minutes rather than hours. Neural networks for tabular data need careful regularization and architecture tuning to match GBDT quality. Choose neural nets when: (a) you have very large datasets (>10M rows), (b) you need to share representations with other modalities, or (c) you need end-to-end fine-tuning with embeddings.

**Q2: Why use logistic regression as a baseline before anything else?**
Logistic regression is linear in features, interpretable, extremely fast to train and serve, and trivial to debug (you can inspect coefficients directly). If it works well, you've saved weeks of ML engineering. If it underperforms, its failure mode tells you whether more data, better features, or a more expressive model is the next step. The baseline doctrine: start simple, fail fast, learn quickly.

**Q3: Why would you use a two-tower model over matrix factorization for retrieval?**
Two-tower models accept arbitrary input features (user context, session features, item content) in addition to IDs, making them far more capable for cold-start and context-aware retrieval. Matrix factorization only uses historical interaction data and produces static embeddings. Two-tower allows online inference of user embeddings based on real-time context, while MF embeddings must be computed offline. The cost: two-tower requires more training data, more compute, and more complex inference infrastructure.

**Q4: Why use random forests over a single decision tree?**
A single decision tree overfits easily (high variance) because it can memorize training data with enough depth. Random forests average many decorrelated trees (each trained on a bootstrap sample with feature subsampling), reducing variance substantially while keeping bias similar. The tradeoff: random forests are 100× more compute at training and inference, but typically 10–30% better on real datasets. Use a single tree only when interpretability of the exact decision path matters.


## Theme 2: Data & Features

**Q5: Why normalize features for gradient descent but not for tree models?**
Gradient descent with unscaled features causes different features to have gradients of very different magnitudes, making the loss landscape elongated and optimization slow. Trees split on individual feature thresholds — the relative scale doesn't affect when a split is good, so normalization doesn't help. Over-normalizing for trees can actually hurt by obscuring natural structure.

**Q6: Why use target encoding over one-hot encoding for high-cardinality categoricals?**
One-hot encoding creates one column per category — for a column with 10K unique values, this is a 10K-dimensional sparse feature that blooms memory and slows training. Target encoding replaces each category with its mean target value (smoothed to prevent leakage), producing a single dense feature. The risk: target encoding can leak label information if not computed on training data only (use leave-one-out or cross-fold target encoding).

**Q7: Why is point-in-time correctness critical for feature engineering?**
If you use features computed with information from after the target label's timestamp during training, the model will learn from the future — leakage. In production, those future features aren't available, so the model performs much worse than offline eval suggested. Example: using a customer's total 30-day spend to predict a purchase that happened on day 15 — the 30-day figure includes the future. Always snapshot features at the moment the prediction would be made.

**Q8: Why use feature hashing instead of a learned embedding for sparse categoricals?**
Feature hashing maps categories to a fixed-size vector without a learned lookup table, using a hash function. This is useful when the vocabulary is dynamic (new users, new products) or extremely large (billions of unique values). A learned embedding table requires a fixed vocabulary and can't handle out-of-vocabulary items. The tradeoff: hashing causes collisions (different items map to the same index) and lacks semantic structure that learned embeddings provide.


## Theme 3: Training & Optimization

**Q9: Why use Adam over SGD for transformer training?**
Transformers have parameters with very different gradient magnitudes — attention weights vs. layer norms vs. feed-forward weights. Adam uses per-parameter adaptive learning rates (via exponential moving averages of gradient magnitudes), naturally handling this heterogeneity. SGD with a single learning rate requires careful tuning to work well. The tradeoff: Adam uses 2× memory (for first and second moment estimates) and can generalize slightly worse on some tasks — some recent research shows SGD can match Adam with enough tuning.

**Q10: Why use batch normalization over layer normalization for transformers?**
You wouldn't — layer norm is standard for transformers, batch norm is standard for CNNs. Batch norm normalizes across the batch dimension, making it depend on batch size and fail at batch size 1 (common at inference). Layer norm normalizes across the feature dimension per sample, making it batch-size independent. Transformers use variable-length sequences and are often used at batch size 1 at inference, so layer norm is the right choice.

**Q11: Why use early stopping instead of a fixed number of epochs?**
Training loss decreases monotonically with more epochs; validation loss eventually starts increasing (overfitting). Early stopping halts training when validation loss hasn't improved for N epochs (patience), keeping the model at its best generalization point. This is both a regularization technique (prevents overfitting) and a compute saver (avoids training longer than necessary).

**Q12: Why is learning rate warmup important for training large models?**
At the start of training, model weights are random and gradients are large and noisy. A large learning rate at initialization causes large, destabilizing updates. Warmup linearly increases the learning rate from ~0 to the target over the first few thousand steps, allowing the optimizer to stabilize before taking large gradient steps. For transformers specifically, warmup is nearly always used (the original "Attention Is All You Need" used 4K warmup steps).


## Theme 4: Evaluation

**Q13: Why use AUC instead of accuracy for imbalanced classification?**
Accuracy is misleading for imbalanced data: a model that always predicts the majority class achieves 99% accuracy on a 99:1 class distribution. AUC measures ranking quality — how well the model separates positives from negatives regardless of the decision threshold. It's threshold-invariant and meaningful even when positive rate is 0.1%. Use precision-recall AUC when the cost of false positives and false negatives are very different (fraud, medical diagnosis).

**Q14: Why use NDCG over precision@k for ranking?**
Precision@k only cares whether items in the top-k are relevant, treating all positions equally. NDCG discounts relevance by position (item at rank 1 is worth more than at rank 10) and supports graded relevance (highly relevant > somewhat relevant > not relevant). For recommendation systems and search, users care more about position — a highly relevant item at rank 10 is worth less than at rank 1.

**Q15: Why use cross-validation instead of a single train/test split?**
A single split is sensitive to which particular samples land in training vs. test — with limited data, the split can be lucky or unlucky. Cross-validation averages performance over multiple folds, giving a lower-variance estimate of generalization performance. The tradeoff: k-fold CV trains k models (k× compute). For large datasets, a single split is usually fine. For small datasets (<10K), k-fold is essential.

**Q16: Why does offline evaluation sometimes fail to predict online performance?**
Offline eval uses a static held-out set and assumes the model is deployed in a world identical to the training world. Online, the model's outputs affect user behavior (feedback loops), the user population may differ from the historical sample (selection bias), and latency/errors may cause some requests to fall back to heuristics. The gap is largest for personalization systems where the model changes what users interact with.


## Theme 5: Production & Serving

**Q17: Why use approximate nearest neighbor (ANN) over exact search at scale?**
Exact nearest neighbor search is O(N × d) per query — for 100M item embeddings at d=128, this is 12.8B multiplications per query, which at 1K QPS is impractical. ANN (HNSW, FAISS) finds the approximate k-nearest in O(log N) with a recall@10 > 95% for most datasets, at the cost of occasionally missing the exact nearest neighbor. In recommendation systems, the difference between the 1st and 10th nearest neighbor is almost never perceptible to users.

**Q18: Why use a feature store instead of computing features on-the-fly?**
Computing features on-the-fly at serving time: (a) may not be possible for features requiring large historical windows (30-day purchase history); (b) couples serving latency to feature computation latency; (c) is likely to introduce training-serving skew if the computation differs from the training pipeline. A feature store precomputes features, ensures consistency between training and serving, and enables low-latency lookups.

**Q19: Why is canary deployment preferred over a hard cutover?**
A hard cutover exposes all users to a potentially degraded model simultaneously — if the model has a bug or unexpected behavior, you've impacted 100% of traffic before you can detect the issue. Canary exposes 1–5% of traffic first, letting you detect problems at small scale with minimal user impact before scaling up. The cost: canary requires traffic-splitting infrastructure and takes longer to fully roll out.

**Q20: Why use gRPC instead of REST for internal ML serving?**
gRPC uses Protocol Buffers (binary serialization) over HTTP/2, which has lower latency and smaller payload size than JSON over HTTP/1.1. For high-throughput internal services (thousands of model inference calls per second), the overhead difference is significant. REST is preferred for external APIs where tooling compatibility (curl, browser, third-party clients) matters more than raw performance.


## Theme 6: LLM Engineering

**Q21: Why use RAG instead of fine-tuning when domain knowledge changes frequently?**
Fine-tuning bakes knowledge into model weights — updating it requires retraining, which takes hours and significant cost. RAG retrieves knowledge at query time from an updatable document store, so new information is available immediately after updating the index. Fine-tune for behavioral changes (output format, tone, style); RAG for knowledge updates. RAG also provides source attribution, which fine-tuning cannot.

**Q22: Why use LoRA instead of full fine-tuning for a 7B model?**
Full fine-tuning requires storing gradients and optimizer states for all 7B parameters — at fp16 + Adam, that's ~84GB just for optimizer states, which requires multiple A100s. LoRA trains only small rank-r adapter matrices (~16–64M parameters), fitting comfortably on a single A100. The quality delta is typically <1% on most tasks, and the trained adapters can be merged into the base model for zero-overhead inference.

**Q23: Why use top-p sampling instead of top-k for text generation?**
Top-k always samples from exactly k tokens regardless of the model's confidence. When the model is confident (peaked distribution), k=50 still includes many low-probability tokens. Top-p adapts: it samples from the smallest set of tokens whose cumulative probability exceeds p. When confident, nucleus is small (2–3 tokens); when uncertain, nucleus is larger. This makes top-p more robust across different confidence levels.

**Q24: Why does quantization to int8 speed up inference?**
Modern hardware has specialized int8 matrix multiplication units (e.g., NVIDIA Tensor Cores in A100) that are 2–4× faster than fp16. INT8 also halves memory footprint vs fp16, allowing larger batches and reducing memory bandwidth pressure. The quality cost is typically <0.5% AUC degradation because neural networks are over-parameterized and robust to small precision reductions. INT4 doubles the benefit again but with larger quality cost.

**Q25: Why use streaming inference (token-by-token output) instead of waiting for the full response?**
Users perceive time-to-first-token and streaming response as much faster than waiting for the complete response. The actual generation time is the same, but perceived latency — which drives user satisfaction — improves dramatically. For a 300-token response at 30 tokens/second, streaming gives the first token in ~33ms; waiting for the full response means 10 seconds before anything appears. Streaming also allows the application to start rendering content before generation completes.


## Common Interview Questions

**Q: When does memorizing these tradeoffs help vs. hurt?**
It helps when the interviewers' question maps cleanly to one of these pairs and your answer demonstrates real understanding. It hurts if you pattern-match to the wrong tradeoff or give a rote answer that doesn't respond to their actual context. Always listen to what they're setting up before answering — "why X over Y in your specific system" is different from "why X over Y in general."

**Q: How should I structure a tradeoff answer?**
One good structure: (1) name the key dimension the tradeoff lives on (speed vs. quality, simplicity vs. expressiveness, training vs. serving cost); (2) explain when X wins on that dimension and why; (3) explain when Y wins and why; (4) if you have a specific context, state which you'd use and why. Aim for 3–4 sentences, not a paragraph.

## Key Takeaways
- GBDT for tabular data (interpretable, fast); neural net when you need representation sharing or scale
- AUC > accuracy for imbalanced; NDCG > precision@k for ranked outputs; PR-AUC for severe imbalance
- Feature store = consistency between training and serving + low-latency precomputed features
- ANN vs. exact search: 0.1% recall loss, 1000× speed gain at scale — almost always worth it
- RAG for facts that change; LoRA fine-tune for behavioral change; prompting as the default start
- Canary > hard cutover; gRPC > REST for internal; int8 quantization for 2× speed with <0.5% loss